In [2]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.preprocessing import OneHotEncoder
from sklearn.model_selection import train_test_split

# Load and preprocess data
iris = load_iris()
X = iris.data
y = iris.target.reshape(-1, 1)
encoder = OneHotEncoder(sparse_output=False)
y_onehot = encoder.fit_transform(y)

# Train/test split
X_train, X_test, y_train, y_test = train_test_split(X, y_onehot, test_size=0.2, random_state=42)

# Network architecture
input_size = X_train.shape[1]
hidden_size = 10
output_size = y_train.shape[1]
lr = 0.1
epochs = 5000

# Initialize weights
W1 = np.random.randn(input_size, hidden_size) * 0.01
b1 = np.zeros((1, hidden_size))
W2 = np.random.randn(hidden_size, output_size) * 0.01
b2 = np.zeros((1, output_size))

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_derivative(z):
    return z * (1 - z)

def softmax(z):
    exp_z = np.exp(z - np.max(z, axis=1, keepdims=True))
    return exp_z / np.sum(exp_z, axis=1, keepdims=True)

for i in range(epochs):
    # Forward pass
    z1 = np.dot(X_train, W1) + b1
    a1 = sigmoid(z1)
    z2 = np.dot(a1, W2) + b2
    a2 = softmax(z2)

    # Backward pass
    error_output = a2 - y_train
    dW2 = np.dot(a1.T, error_output) / X_train.shape[0]
    db2 = np.sum(error_output, axis=0, keepdims=True) / X_train.shape[0]

    error_hidden = np.dot(error_output, W2.T)
    dW1 = np.dot(X_train.T, error_hidden * sigmoid_derivative(a1)) / X_train.shape[0]
    db1 = np.sum(error_hidden * sigmoid_derivative(a1), axis=0, keepdims=True) / X_train.shape[0]

    # Update weights
    W2 -= lr * dW2
    b2 -= lr * db2
    W1 -= lr * dW1
    b1 -= lr * db1

    if i % 500 == 0:
        loss = -np.mean(np.sum(y_train * np.log(a2 + 1e-8), axis=1))
        print(f"Epoch {i}, Loss: {loss:.4f}")

# Prediction and accuracy
def predict(X):
    a1 = sigmoid(np.dot(X, W1) + b1)
    a2 = softmax(np.dot(a1, W2) + b2)
    return np.argmax(a2, axis=1)

y_pred = predict(X_test)
y_true = np.argmax(y_test, axis=1)
accuracy = np.mean(y_pred == y_true) * 100
print(f"Test Accuracy: {accuracy:.2f}%")

Epoch 0, Loss: 1.0986
Epoch 500, Loss: 0.3223
Epoch 1000, Loss: 0.1479
Epoch 1500, Loss: 0.1044
Epoch 2000, Loss: 0.0875
Epoch 2500, Loss: 0.0788
Epoch 3000, Loss: 0.0735
Epoch 3500, Loss: 0.0699
Epoch 4000, Loss: 0.0674
Epoch 4500, Loss: 0.0655
Test Accuracy: 100.00%
